# 1. Optimizer

A ready-made **class** that holds references to the model's parameters.

It reads each parameter's gradient (`p.grad`) and uses it to **update the parameter's value**. The `backward()` computes the gradients, and the optimizer applies them to the weights.

---
# 2. Learning Rate

The gradient says **which direction** to move each weight. The learning rate (`lr`) says **how far** to move in that direction.

---

# 3. TYPES


## 3.1 Sthocastic Gradient Descent (SGD)

$w = w - lr · grad$


```python

sgd = torch.optim.SGD(
    params = model.parameters() # The params of the model
    lr = 0.001, # The value of LEARNING RATE
    weight_decay = 1e-3 # The value of WEIGHT DECAY

)

```

### 3.1.1 Weight Decay

a technique to regularization, with this option we have some news, the formula will be

$w = w - (lr * (grad + (decay * grada)))$

In [4]:
import torch
import torch.nn as nn

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
print(f'Before the training: {model.weight.item()}')

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])      # y = 2x

for epoch in range(150):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

print(f'After the training: {model.weight.item()}')    # ~2.0

Before the training: 0.8485426902770996
After the training: 1.9987268447875977


## 3.2 Momentum

Momentum keeps a **running memory of previous gradients**, allowing the optimizer to build up velocity in consistent directions.

The update can be written as:

$$
v_t = \beta v_{t} + g_t
$$

$$
w_t = w_{t} - lr \cdot v_t
$$

Where:

* $w$ = model weight
* $g_t$ = current gradient
* $v_t$ = accumulated velocity
* $\beta$ = momentum coefficient
* $lr$ = learning rate

Substituting $v_t$ into the weight update:

$$
w_t = w_{t} - lr \cdot (\beta v_{t} + g_t)
$$

The idea is that the optimizer does not use only the **current gradient**. It also considers the **direction of previous updates**.

### PyTorch

```python
momentum = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.8  # β: momentum coefficient
)
```
With `momentum=0.8`, approximately 80% of the previous velocity is retained at each step, while the current gradient contributes to the new velocity.



> **Momentum = gradient + memory of previous movement.**


In [8]:
import torch
import torch.nn as nn

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum = 0.7)
print(f'Before the training: {model.weight.item()}')

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])      # y = 2x

for epoch in range(150):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

print(f'After the training: {model.weight.item()}')    # ~2.0

Before the training: -0.32069969177246094
After the training: 2.0


## 3.3 RMSProp

RMSProp keeps a **moving average of squared gradients**. This allows the optimizer to adapt the learning rate for each parameter based on the magnitude of its recent gradients.

First, it keeps a running average of the squared gradients:

$$
v_t = \alpha v_{t-1} + (1-\alpha)g_t^2
$$

Then, the parameters are updated as:

$$
w_t = w_{t-1} - lr \cdot \frac{g_t}{\sqrt{v_t}+\epsilon}
$$

Where:

* $w$ = model weight
* $g_t$ = current gradient
* $v_t$ = moving average of squared gradients
* $\alpha$ = decay rate of the moving average
* $lr$ = learning rate
* $\epsilon$ = small value used for numerical stability

### Why does RMSProp help?

Suppose a parameter consistently receives large gradients:

$$
g_t \rightarrow \text{large}
$$

Then:

$$
v_t \rightarrow \text{large}
$$

So:

$$
\frac{g_t}{\sqrt{v_t}+\epsilon}
\rightarrow
\text{smaller update}
$$

For parameters with smaller gradients, the denominator tends to be smaller, resulting in a relatively larger update.

So the basic idea is:

> **RMSProp automatically adjusts the effective step size for each parameter according to the recent magnitude of its gradients.**

### PyTorch

```python
optimizer = torch.optim.RMSprop(
    model.parameters(),
    lr=0.001,
    alpha=0.99,
    eps=1e-8,
    weight_decay=0,
    momentum=0,
    centered=False
)
```

### Hyperparameters

| Hyperparameter | Meaning                                                 |
| -------------- | ------------------------------------------------------- |
| `lr`           | Learning rate                                           |
| `alpha`        | Decay factor of the moving average of squared gradients |
| `eps`          | Numerical stability term                                |
| `weight_decay` | L2 regularization / weight decay                        |
| `momentum`     | Adds momentum to the RMSProp update                     |
| `centered`     | Uses a centered estimate of the gradient variance       |


So:

> **RMSProp = adapt the step size according to the recent magnitude of the gradients.**


In [12]:
import torch
import torch.nn as nn

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.1, alpha=0.99,)
print(f'Before the training: {model.weight.item()}')

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])      # y = 2x

for epoch in range(150):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

print(f'After the training: {model.weight.item()}')    # ~2.0

Before the training: 0.8054791688919067
After the training: 1.9998992681503296


## 3.4 Adam Family (Adam, AdamW)

Adam (**Adaptive Moment Estimation**) combines the main ideas of **Momentum** and **RMSProp**.

Instead of keeping only one type of information about the gradient, Adam keeps two:

* **First moment ($m_t$)** → tracks the direction of the gradients, similar to Momentum.
* **Second moment ($v_t$)** → tracks the magnitude of squared gradients, similar to RMSProp.

### Adam

First, Adam updates the first moment:

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t
$$

Then, the second moment:

$$
v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2
$$

Because both moments start at zero, Adam applies **bias correction**:

$$
\hat{m}_t = \frac{m_t}{1-\beta_1^t}
$$

$$
\hat{v}_t = \frac{v_t}{1-\beta_2^t}
$$

Finally, the parameters are updated:

$$
w_t =
w_{t-1}
-
lr
\cdot
\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
$$

So Adam combines:

```text
Momentum
    ↓
First moment (direction)
    +
RMSProp
    ↓
Second moment (magnitude)
    ↓
Adaptive update
```

### PyTorch

```python
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0
)
```

### Adam Hyperparameters

| Hyperparameter | Meaning                                      |        Default |
| -------------- | -------------------------------------------- | -------------: |
| `lr`           | Learning rate                                |         `1e-3` |
| `betas`        | Decay rates for the first and second moments | `(0.9, 0.999)` |
| `eps`          | Numerical stability term                     |         `1e-8` |
| `weight_decay` | Weight decay / L2 penalty in Adam            |            `0` |
| `amsgrad`      | Uses the AMSGrad variant                     |        `False` |

In `betas=(β₁, β₂)`:

$$
\beta_1 \rightarrow \text{memory of gradient direction}
$$

$$
\beta_2 \rightarrow \text{memory of squared gradient magnitude}
$$

---

## AdamW

AdamW keeps the same adaptive mechanism as Adam, but **decouples weight decay from the gradient update**.

In AdamW, the weight decay is applied directly to the weights:

$$
w_t
=
w_{t-1}
-
lr
\cdot
\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
-
lr\cdot\lambda w_{t-1}
$$

Where:

* $\lambda$ = `weight_decay`

The key difference is:

```text
Adam
gradient
   ↓
gradient + weight decay
   ↓
Adam update
```

while AdamW does:

```text
gradient
   ↓
Adam update
   ↓
+
weight decay applied separately
```

This is why AdamW is generally preferred when using weight decay. PyTorch's current implementation explicitly defines AdamW so that weight decay does not accumulate into the momentum or variance terms.

### PyTorch

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01
)
```

### AdamW Hyperparameters

| Hyperparameter | Meaning                                  |        Default |
| -------------- | ---------------------------------------- | -------------: |
| `lr`           | Learning rate                            |         `1e-3` |
| `betas`        | Decay rates for first and second moments | `(0.9, 0.999)` |
| `eps`          | Numerical stability term                 |         `1e-8` |
| `weight_decay` | Decoupled weight decay                   |         `0.01` |
| `amsgrad`      | Uses AMSGrad                             |        `False` |

### Adam vs AdamW

```text
Adam
├── First moment → direction
├── Second moment → magnitude
└── Optional weight decay coupled to gradient

AdamW
├── First moment → direction
├── Second moment → magnitude
└── Weight decay → applied separately
```

So the simplest mental model is:

> **Adam = Momentum + RMSProp**

> **AdamW = Adam + decoupled Weight Decay**


In [47]:
import torch
import torch.nn as nn

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
print(f'Before the training: {model.weight.item()}')

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])      # y = 2x

for epoch in range(150):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

print(f'After the training: {model.weight.item()}')    # ~2.0

Before the training: -0.633446216583252
After the training: 1.7086864709854126


In [40]:
import torch
import torch.nn as nn

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1)
print(f'Before the training: {model.weight.item()}')

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])      # y = 2x

for epoch in range(150):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

print(f'After the training: {model.weight.item()}')    # ~2.0

Before the training: 0.7845687866210938
After the training: 1.9575395584106445


## 3.5 Muon

Muon (**MomentUm Orthogonalized by Newton-schulz**) is an optimizer designed primarily for **2D weight matrices in neural network hidden layers**, especially the matrices used by `Linear` layers.

Unlike Adam and AdamW, which adapt the update element-by-element, Muon uses **momentum followed by Newton-Schulz orthogonalization** to transform the update matrix.

### Momentum

First, Muon builds a momentum buffer:

$$
B_t = \mu B_{t} + g_t
$$

Where:

* $B_t$ = momentum buffer
* $g_t$ = current gradient
* $\mu$ = momentum coefficient

Muon can also use **Nesterov momentum**:

$$
\widetilde{B}_t = g_t + \mu B_t
$$

when `nesterov=True`.

### Newton-Schulz Orthogonalization

After obtaining the momentum update, Muon applies a few **Newton-Schulz iterations** to approximately orthogonalize the update matrix:

$$
O_t = \operatorname{NS}_k(\widetilde{B}_t)
$$

The goal is to transform the momentum matrix into an update with a more useful matrix structure rather than directly applying the raw gradient.


The final parameter update also uses **decoupled weight decay**:

$$
W_t =
W_{t}
-
lr \cdot \lambda W_{t}
-
lr \cdot O_t
$$

So Muon has:

```text
Momentum
   +
Newton-Schulz orthogonalization
   +
Weight decay
```

### PyTorch

```python
optimizer = torch.optim.Muon(
    model.parameters(),
    lr=0.02,
    momentum=0.95,
    weight_decay=0.1,
    nesterov=True,
    ns_steps=5,
    eps=1e-7
)
```

However, Muon is intended for **2D hidden-layer parameters**. Parameters such as **biases and embeddings** should generally be optimized with a standard optimizer such as AdamW.

### Muon Hyperparameters

| Hyperparameter    | Meaning                                       |            PyTorch default |
| ----------------- | --------------------------------------------- | -------------------------: |
| `lr`              | Learning rate                                 |                     `1e-3` |
| `momentum`        | Momentum coefficient                          |                     `0.95` |
| `weight_decay`    | Decoupled weight decay                        |                      `0.1` |
| `nesterov`        | Enables Nesterov momentum                     |                     `True` |
| `ns_steps`        | Number of Newton-Schulz iterations            |                        `5` |
| `eps`             | Numerical stability term                      |                     `1e-7` |
| `ns_coefficients` | Coefficients of the Newton-Schulz polynomial  | `(3.4445, -4.775, 2.0315)` |
| `adjust_lr_fn`    | Controls shape-based learning-rate adjustment |                     `None` |

PyTorch currently provides several learning-rate adjustment modes for rectangular matrices, including `"original"`, `"match_rms_adamw"`, and `"spectral_unclamped"`.

So:

> **AdamW = adaptive optimization of individual parameters**

> **Muon = momentum + matrix orthogonalization, designed for hidden-layer weight matrices**


In [49]:
import torch
import torch.nn as nn

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.Muon([model.weight], lr=0.1)
print(f'Before the training: {model.weight.item()}')

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])      # y = 2x

for epoch in range(150):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

print(f'After the training: {model.weight.item()}')    # ~2.0

Before the training: -0.8380724191665649
After the training: 1.9519213438034058


---

# 4. Split  Weights into different optimizers

A common setup is therefore:

```text
Linear.weight       → Muon
Embedding.weight    → AdamW
LayerNorm.weight    → AdamW
LayerNorm.bias      → AdamW
Linear.bias         → AdamW
```


In [29]:
import torch
import torch.nn as nn

# Create the model
class Test(nn.Module):
  def __init__(self, d_model = 10, vocab_size = 100):
    super().__init__()


    self.embedding = nn.Embedding(vocab_size, d_model)
    self.wQ = nn.Linear(d_model, d_model)
    self.wK = nn.Linear(d_model, d_model)
    self.wV = nn.Linear(d_model, d_model)


    def forward(self, x):

        x = self.wQ(x)

        return x


model = Test()
 
for module in model.modules():
    print(module)

# Split the weights for each optmizer

muon_params = []
adamw_params = []

for module in model.modules():

    # Linear Weights
    if isinstance(module, nn.Linear):
        muon_params.append(module.weight)

        # Linear Bias
        if module.bias is not None:
            adamw_params.append(module.bias)

    # Embedding Weights
    elif isinstance(module, nn.Embedding):
        adamw_params.append(module.weight)

    # Norm Weights
    elif isinstance(module, (nn.LayerNorm,)):
        adamw_params.append(module.weight)

        # Norm Bias
        if module.bias is not None:
            adamw_params.append(module.bias)

# Create the optimizers
optimizer_muon = torch.optim.Muon(
    muon_params,
    lr=0.02
)

optimizer_adamw = torch.optim.AdamW(
    adamw_params,
    lr=1e-4,
    weight_decay=0.01
)

Test(
  (embedding): Embedding(100, 10)
  (wQ): Linear(in_features=10, out_features=10, bias=True)
  (wK): Linear(in_features=10, out_features=10, bias=True)
  (wV): Linear(in_features=10, out_features=10, bias=True)
)
Embedding(100, 10)
Linear(in_features=10, out_features=10, bias=True)
Linear(in_features=10, out_features=10, bias=True)
Linear(in_features=10, out_features=10, bias=True)
